# Reviewer Lookup Table Generator

This notebook generates a `reviewer_stats.json` file from the Amazon training data.

The JSON file contains:
- Population level statistics (median, mode, mean) used as fallback defaults at inference time
- A reviewer-level lookup table mapping each `reviewerID` to their historical review frequency

This  solves the training serving skew problem where `review_frequency` is available during 
training but not at inference time. At serving time, the app looks up the reviewer ID and 
uses their real frequency. If not found, it falls back to the population median.

In [1]:
# Install neccessary Libraries
!pip install pandas

In [2]:
# Import necessary libraries

import pandas as pd
import json

# Load the training data

In [ ]:


df = pd.read_csv("Amazon_review_final.csv")

## Generate Reviewer Statistics

This function takes the training dataframe and computes two things:

1. **Population statistics**: the median, mode, and mean of `review_frequency` across all reviewers. These are used as fallback defaults at inference time when a reviewer is not found in the lookup table.

2. **Reviewer lookup table**: a dictionary mapping each `reviewerID` to their exact review frequency from the training data. This allows the app to serve the real historical value instead of an estimate.

The output is saved as `reviewer_stats.json` and loaded by the web app at startup.

In [4]:
def generate_reviewer_stats(df: pd.DataFrame, output_path: str = "reviewer_stats.json"):
    """
    Computes reviewer level statistics from the training dataframe
    and saves them to a JSON file for use at inference time.

    Parameters:
        df: Training dataframe containing 'reviewerID' column.
        output:  Where to save the JSON file.
    """

    # Recompute review_frequency the same way it was done during training
    df["review_frequency"] = df.groupby("reviewerID")["reviewerID"].transform("count")

    # Calculate median, mode, and mean review frequency across all reviewers as fallbacks
    median_freq = int(df["review_frequency"].median())
    mode_freq   = int(df["review_frequency"].mode()[0])
    mean_freq   = round(float(df["review_frequency"].mean()), 4)

    # Create a lookup dictionary mapping reviewerID to their review frequency
    reviewer_lookup = (
        df.groupby("reviewerID")["review_frequency"]
        .first()
        .astype(int)
        .to_dict()
    )

    stats = {
        "median":          median_freq,
        "mode":            mode_freq,
        "mean":            mean_freq,
        "reviewer_lookup": reviewer_lookup
    }

    with open(output_path, "w") as f:
        json.dump(stats, f, indent=2)

    print(f"reviewer_stats.json saved to: {output_path}")
    print(f"  Median : {median_freq}")
    print(f"  Mode   : {mode_freq}")
    print(f"  Mean   : {mean_freq}")
    print(f"  Reviewers in lookup: {len(reviewer_lookup):,}")

    return stats


## Load and Save

Loads the training data, calls the function above, and saves `reviewer_stats.json`.

In [5]:
if __name__ == "__main__":
    df = pd.read_csv("Amazon_review_final.csv")
    generate_reviewer_stats(df)
    print("Lookup generated.")


reviewer_stats.json saved to: reviewer_stats.json
  Median : 1
  Mode   : 1
  Mean   : 1.641
  Reviewers in lookup: 630,543
Lookup generated.
